In [4]:
!pip install streamlit
!pip install reportlab
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from reportlab.lib.pagesizes import LETTER
from reportlab.pdfgen import canvas
import tempfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.1 MB/s eta 0:00:00


In [5]:
# CONFIG
st.set_page_config(
    page_title="Supplement Scheduler",
    layout="wide"
)

2026-04-24 17:08:46.803 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [6]:
# LOAD DATA
@st.cache_data
def load_data():
    return pd.read_excel("/content/Vitamin_RuleModel_Output (6).xlsx")

df = load_data()

st.title("💊 Supplement Scheduler")
st.caption("Rule‑based scheduling using manufacturer directions")

2026-04-24 17:09:02.471 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-24 17:09:02.474 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-24 17:09:02.474 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:12:49.476 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:12:49.859 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-24 17:12:49.859 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:12:49.860 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running 

DeltaGenerator()

In [7]:
# USER PROFILE
st.sidebar.header("Your Daily Routine")

wake_time = st.sidebar.time_input("Wake time", value=datetime.strptime("07:00", "%H:%M").time())
breakfast = st.sidebar.time_input("Breakfast", value=datetime.strptime("08:00", "%H:%M").time())
lunch = st.sidebar.time_input("Lunch", value=datetime.strptime("13:00", "%H:%M").time())
dinner = st.sidebar.time_input("Dinner", value=datetime.strptime("19:00", "%H:%M").time())
bed_time = st.sidebar.time_input("Bedtime", value=datetime.strptime("22:30", "%H:%M").time())

2026-04-24 17:13:04.645 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.647 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.648 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.650 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.651 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.652 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:04.653 Session state does not function when running a script without `streamlit run`
2026-04-24 17:13

In [8]:
# SUPPLEMENT SELECTION
st.header("Select Supplements")

supplement_names = sorted(df["Product Name"].dropna().unique())
chosen = st.multiselect("Choose supplements", supplement_names)

chosen_df = df[df["Product Name"].isin(chosen)].copy()

2026-04-24 17:13:10.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.235 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.236 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.256 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.299 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:10.300 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [9]:
# HELPERS
def compute_timing(row):
    if row.get("bedtime"):
        return "Bedtime"
    if row.get("morning"):
        return "Morning"
    if row.get("empty_stomach"):
        return "Empty stomach"
    if row.get("with_food"):
        return "With meals"
    return "Anytime"

def timing_to_clock(timing):
    if timing == "Morning":
        return wake_time
    if timing == "With meals":
        return breakfast
    if timing == "Empty stomach":
        return (datetime.combine(datetime.today(), breakfast) - timedelta(minutes=30)).time()
    if timing == "Bedtime":
        return (datetime.combine(datetime.today(), bed_time) - timedelta(minutes=30)).time()
    return wake_time


In [10]:
# CYCLE LOGIC
st.header("Cycle Settings")

use_cycle = st.checkbox("Enable cycle (30 days on / 7 days off)", value=True)
cycle_start = st.date_input("Cycle start date", datetime.today())

def is_on_cycle(date):
    if not use_cycle:
        return True
    delta = (date - cycle_start).days
    cycle_day = delta % 37
    return cycle_day < 30

today = datetime.today().date()
on_cycle_today = is_on_cycle(today)

if not on_cycle_today:
    st.warning("❌ Today is an OFF‑cycle day. No supplements scheduled.")
    st.stop()

2026-04-24 17:13:15.743 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.745 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.747 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.748 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.749 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:15.749 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [11]:
# DOSING CONTROLS
st.header("Dosage & Timing")

schedule_rows = []

for _, row in chosen_df.iterrows():
    st.subheader(row["Product Name"])

    min_dose = row.get("daily_dose_min", 1)
    max_dose = row.get("daily_dose_max", min_dose)

    dose = st.number_input(
        "Daily dose",
        min_value=float(min_dose),
        max_value=float(max_dose),
        value=float(min_dose),
        key=f"dose_{row['Product Name']}"
    )

    timing = compute_timing(row)
    time_of_day = timing_to_clock(timing)

    if row.get("max_warning"):
        st.warning("⚠ Do not exceed recommended daily amount")

    if row.get("caffeine_warning"):
        st.warning("⚠ Avoid caffeine")

    schedule_rows.append({
        "Supplement": row["Product Name"],
        "Dose": dose,
        "Timing": timing,
        "Time": time_of_day.strftime("%H:%M"),
        "Instructions": row.get("suggested_use_clean", "")
    })

2026-04-24 17:13:18.384 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:18.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:13:18.387 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [14]:
# SCHEDULE DISPLAY
schedule_df = pd.DataFrame(schedule_rows)
if not schedule_df.empty:
    schedule_df = schedule_df.sort_values("Time")

st.header("📅 Today's Schedule")
st.dataframe(schedule_df, width='stretch')

2026-04-24 17:14:01.072 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:01.075 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:01.075 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:01.077 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:01.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:01.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [15]:
# PDF GENERATION
def generate_pdf(schedule_df):
    file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    c = canvas.Canvas(file.name, pagesize=LETTER)
    width, height = LETTER

    c.setFont("Helvetica-Bold", 16)
    c.drawString(40, height - 40, "Daily Supplement Schedule")

    c.setFont("Helvetica", 10)
    y = height - 80

    for _, row in schedule_df.iterrows():
        text = f"{row['Time']} — {row['Supplement']} ({row['Dose']})"
        c.drawString(40, y, text)
        y -= 15

        if row["Instructions"]:
            c.setFont("Helvetica-Oblique", 9)
            c.drawString(60, y, row["Instructions"][:90])
            c.setFont("Helvetica", 10)
            y -= 20

        if y < 80:
            c.showPage()
            y = height - 80

    c.save()
    return file.name

st.header("📄 Export")

if st.button("Download PDF Schedule"):
    pdf_path = generate_pdf(schedule_df)
    with open(pdf_path, "rb") as f:
        st.download_button(
            "Download PDF",
            f,
            file_name="supplement_schedule.pdf",
            mime="application/pdf"
        )

2026-04-24 17:14:20.800 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.802 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.803 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.804 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.805 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.806 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.808 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:20.809 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [16]:
# DISCLAIMER
st.divider()
st.caption(
    "⚠ This tool follows manufacturer label instructions only. "
    "It does not provide medical advice. Consult a healthcare professional before use."
)

2026-04-24 17:14:23.825 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:23.826 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:23.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:23.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:23.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 17:14:23.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [18]:
%%writefile app.py
!streamlit run app.py


Writing app.py
